# Arabizi → Arabic: نسخة أولية كاملة (Pilot) في 3 أيام
**الهدف:** قياس تدهور BM25 وmE5 والهجين مع استعلامات Arabizi، ثم إصلاحها بتحويل عكسي بقيد معجمي ودمج مرشحات (RRF).

**تنبيه علمي:** هذه نسخة تجريبية (200 استعلام، مدوّنة مصغّرة، مجموعة بشرية صغيرة). النتائج *مؤشرات أولية* لا ادعاءات نهائية.

**طريقة الاستخدام:** شغّلي الخلايا بالترتيب. تحتاجين GPU (Runtime → Change runtime type → T4) للجزء Dense فقط.
الحالات في التصميم 2×2: Q0 فصحى عربي | Q1 فصحى Arabizi (مولَّد) | Q2 لهجة عربي (بشري) | Q3 لهجة Arabizi (بشري).

In [ ]:
!pip -q install rank_bm25 sentence-transformers huggingface_hub

In [ ]:
import os, re, gzip, json, random, csv, math
from collections import Counter
from functools import lru_cache
import numpy as np, pandas as pd

SEED = 42
N_QUERIES = 200            # عدد استعلامات dev
N_RANDOM_PASSAGES = 30000  # فقرات عشوائية تُضاف للفقرات المحكومة
USE_DENSE = True           # False لتشغيل BM25 فقط (أسرع للتجارب الأولى)
K_CANDIDATES = 5           # عدد مرشحات التحويل
N_HUMAN = 60               # عدد الاستعلامات التي ستكتبينها بنفسك
OUT = "/content/drive/MyDrive/arabizi_pilot" if os.path.exists("/content/drive") else "./arabizi_pilot"
os.makedirs(OUT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

In [ ]:
# اختياري: ربط Google Drive لحفظ النتائج
from google.colab import drive
drive.mount('/content/drive')
OUT = "/content/drive/MyDrive/arabizi_pilot"; os.makedirs(OUT, exist_ok=True)

Mounted at /content/drive


## 1) البيانات: MIRACL-ar (dev)
نأخذ N استعلامًا، ونحتفظ بكل الفقرات المحكومة لها (إيجابية وسلبية) + عيّنة عشوائية من باقي المدوّنة.
**ملاحظة:** أسماء ملفات المستودع تُستخرج برمجيًا (`list_repo_files`) لأني لم أستطع فتح المستودع من بيئتي. إن تغيّرت البنية فاطبعي `files` وعدّلي الفلترة.
**مهم:** الأرقام المطلقة ستختلف عن المنشور لأن المدوّنة مصغّرة. المقارنة نسبية بين الحالات.

In [ ]:
from huggingface_hub import list_repo_files, hf_hub_download
files = list_repo_files("miracl/miracl", repo_type="dataset")
topics_f = [f for f in files if "miracl-v1.0-ar" in f and "/topics/" in f and "dev" in f][0]
qrels_f  = [f for f in files if "miracl-v1.0-ar" in f and "/qrels/"  in f and "dev" in f][0]
topics_path = hf_hub_download("miracl/miracl", topics_f, repo_type="dataset")
qrels_path  = hf_hub_download("miracl/miracl", qrels_f,  repo_type="dataset")

all_topics, all_qrels = {}, {}
for line in open(topics_path, encoding="utf-8"):
    qid, q = line.rstrip("\n").split("\t", 1); all_topics[qid] = q
for line in open(qrels_path, encoding="utf-8"):
    qid, _, docid, rel = line.split()
    all_qrels.setdefault(qid, {})[docid] = int(rel)

cand = sorted(q for q in all_topics if any(r > 0 for r in all_qrels.get(q, {}).values()))
random.Random(SEED).shuffle(cand)
sel = cand[:N_QUERIES]
queries = {q: all_topics[q] for q in sel}
qrels   = {q: {d for d, r in all_qrels[q].items() if r > 0} for q in sel}
judged  = {d for q in sel for d in all_qrels[q]}

cfiles = sorted(f for f in list_repo_files("miracl/miracl-corpus", repo_type="dataset")
                if "miracl-corpus-v1.0-ar" in f and f.endswith(".jsonl.gz"))
P_KEEP = N_RANDOM_PASSAGES / 2061414
rr = random.Random(SEED); corpus = {}
for f in cfiles:
    p = hf_hub_download("miracl/miracl-corpus", f, repo_type="dataset")
    with gzip.open(p, "rt", encoding="utf-8") as fh:
        for line in fh:
            d = json.loads(line)
            if d["docid"] in judged or rr.random() < P_KEEP:
                corpus[d["docid"]] = (d.get("title", "") + " " + d["text"]).strip()
print(len(queries), "queries |", len(corpus), "passages")

topics.miracl-v1.0-ar-dev.tsv:   0%|          | 0.00/175k [00:00<?, ?B/s]

qrels.miracl-v1.0-ar-dev.tsv:   0%|          | 0.00/573k [00:00<?, ?B/s]

miracl-corpus-v1.0-ar/docs-0.jsonl.gz: reconstructing file:   0%|          |  0.00B / 94.1MB            

miracl-corpus-v1.0-ar/docs-0.jsonl.gz: downloading bytes:           |  0.00B            

miracl-corpus-v1.0-ar/docs-1.jsonl.gz: reconstructing file:   0%|          |  0.00B / 83.8MB            

...

miracl-corpus-v1.0-ar/docs-3.jsonl.gz: reconstructing file:   0%|          |  0.00B / 64.6MB            

miracl-corpus-v1.0-ar/docs-3.jsonl.gz: downloading bytes:           |  0.00B            

miracl-corpus-v1.0-ar/docs-4.jsonl.gz: reconstructing file:   0%|          |  0.00B / 7.23MB            

miracl-corpus-v1.0-ar/docs-4.jsonl.gz: downloading bytes:           |  0.00B            

200 queries | 31909 passages


In [ ]:
qid = list(queries)[0]
print("السؤال:", queries[qid])


السؤال: متى حصلت مجزرة "جمعة الكرامة" ؟


In [ ]:
print("الفقرات الصحيحة:", qrels[qid])
first = list(qrels[qid])[0]
print("مثال فقرة صحيحة:", corpus[first][:300])

الفقرات الصحيحة: {'1237559#7', '1029209#11'}
مثال فقرة صحيحة: الفرقة الاولى مدرع (اليمن) وقعت مجزرة جمعة الكرامة بالطرف الجنوبي من ساحة التغيير بصنعاء، الذي كان وقتها ساحة للمتظاهرين المعارضين لعلي عبد الله صالح. مع انتهاء عشرات الآلاف من المتظاهرين من صلاة الظهر، بدأ مسلحون ملثمون في إطلاق النار عليهم من الشارع، ومن فوق الأشجار، ومن أسطح المنازل، بما فيها منز


## 2) المعالجة، BM25، Dense، الدمج، المقاييس
**قاعدة اتساق:** نفس دالة التطبيع على الفهرس وعلى الاستعلام (درس ورقة MDPI: عدم الاتساق يضر). هذا BM25 بسيط (rank_bm25) لا محلل Lucene العربي، فاذكري ذلك في الورقة.

In [ ]:
AR_DIAC = re.compile(r'[\u064B-\u0652\u0640\u0670]')
def norm_ar(s):
    s = AR_DIAC.sub('', s)
    s = re.sub('[أإآٱ]', 'ا', s)
    s = s.replace('ى', 'ي').replace('ة', 'ه').replace('ؤ', 'و').replace('ئ', 'ي')
    return s.lower()
TOKEN = re.compile(r'\w+', re.UNICODE)
def tok(s): return TOKEN.findall(norm_ar(s))

from rank_bm25 import BM25Okapi
doc_ids = list(corpus)
doc_tokens = [tok(corpus[d]) for d in doc_ids]
bm25 = BM25Okapi(doc_tokens)

@lru_cache(maxsize=None)
def bm25_search(q, k=100):
    t = tok(q)
    if not t: return ()
    s = bm25.get_scores(t)
    if s.max() <= 0: return ()          # لا تطابق = لا نتائج (حالة Arabizi الخام)
    kk = min(k, len(s) - 1)
    idx = np.argpartition(-s, kk)[:kk]
    idx = idx[np.argsort(-s[idx])]
    return tuple(doc_ids[i] for i in idx if s[i] > 0)

In [ ]:
if USE_DENSE:
    import torch
    from sentence_transformers import SentenceTransformer
    enc = SentenceTransformer("intfloat/multilingual-e5-small",
                              device="cuda" if torch.cuda.is_available() else "cpu")
    enc.max_seq_length = 256
    PEMB = enc.encode(["passage: " + corpus[d] for d in doc_ids], batch_size=128,
                      normalize_embeddings=True, show_progress_bar=True, convert_to_numpy=True)
    @lru_cache(maxsize=None)
    def dense_search(q, k=100):
        v = enc.encode(["query: " + q], normalize_embeddings=True, convert_to_numpy=True)[0]
        s = PEMB @ v
        kk = min(k, len(s) - 1)
        idx = np.argpartition(-s, kk)[:kk]
        idx = idx[np.argsort(-s[idx])]
        return tuple(doc_ids[i] for i in idx)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/250 [00:00<?, ?it/s]

In [ ]:
def rrf(runs, k=60, depth=100):
    scores = {}
    for run in runs:
        for rank, d in enumerate(run[:depth], start=1):
            scores[d] = scores.get(d, 0.0) + 1.0 / (k + rank)
    return tuple(sorted(scores, key=scores.get, reverse=True))

def search(system, q):
    if system == "bm25":   return bm25_search(q)
    if system == "dense":  return dense_search(q)
    if system == "hybrid": return rrf([bm25_search(q), dense_search(q)])
    raise ValueError(system)

def ndcg10(ranked, rel):
    dcg = sum(1.0 / math.log2(i + 2) for i, d in enumerate(ranked[:10]) if d in rel)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(rel), 10)))
    return dcg / idcg if idcg > 0 else 0.0

def paired_bootstrap(a, b, n=10000, seed=SEED):
    rng = np.random.default_rng(seed)
    d = np.asarray(a, float) - np.asarray(b, float)
    means = d[rng.integers(0, len(d), size=(n, len(d)))].mean(axis=1)
    lo, hi = np.percentile(means, [2.5, 97.5])
    p = min(1.0, 2 * min((means <= 0).mean(), (means >= 0).mean()))
    return d.mean(), (lo, hi), p

def holm(pvals):
    m = len(pvals); order = np.argsort(pvals); adj = np.empty(m); run = 0.0
    for r, i in enumerate(order):
        run = max(run, (m - r) * pvals[i]); adj[i] = min(1.0, run)
    return adj

## 3) مولّد Arabizi الاصطناعي (3 مستويات ضجيج)
L1 شبه منتظم (أقرب لرومنة Chari الآلية: بلا أرقام ولا حذف حركات) | L2 متوسط | L3 ضجيج عالٍ (أرقام كثيرة وحذف حركات).
**تحذير الدائرية:** جدول المحوّل (القسم 4) كُتب منفصلًا عن هذا الجدول، لكنهما يعكسان أعرافًا متشابهة، فالمجموعة البشرية هي الاختبار غير المتحيز.

In [ ]:
ARABIZI = {
 'ا':['a','aa','e',''], 'أ':['a','2'], 'إ':['e','i','2'], 'آ':['aa','2a'], 'ٱ':['a'],
 'ب':['b'], 'ت':['t'], 'ث':['th','s','t'], 'ج':['j','g'], 'ح':['7','h'], 'خ':['5','kh'],
 'د':['d'], 'ذ':['th','z','d'], 'ر':['r'], 'ز':['z'], 'س':['s'], 'ش':['sh','ch'],
 'ص':['s','9'], 'ض':['d','9'], 'ط':['t','6'], 'ظ':['z','th','6'], 'ع':['3','a','aa'],
 'غ':['gh','3'], 'ف':['f'], 'ق':['q','2','9','k'], 'ك':['k'], 'ل':['l'], 'م':['m'], 'ن':['n'],
 'ه':['h','a','e'], 'و':['w','o','u','oo'], 'ي':['y','i','e','ee'], 'ى':['a','y'],
 'ة':['a','e','h'], 'ء':['2','a',''], 'ؤ':['2','o'], 'ئ':['2','e'],
}
VOWELS = {'a','e','i','o','u','aa','ee','oo'}
LEVELS = {1: (0.0, 0.0), 2: (0.5, 0.3), 3: (0.9, 0.6)}   # (p_digit, p_drop_vowel)

def to_arabizi(text, r, p_digit=0.5, p_drop=0.3):
    words = []
    for w in text.split():
        out = []
        for ch in AR_DIAC.sub('', w):
            if ch not in ARABIZI:
                out.append(ch.lower() if ch.isascii() else ''); continue
            opts = ARABIZI[ch]
            digs = [o for o in opts if o.isdigit()]
            lets = [o for o in opts if not o.isdigit()]
            if digs and (not lets or r.random() < p_digit): o = r.choice(digs)
            else: o = r.choice(lets)
            if o in VOWELS and r.random() < p_drop: o = ''
            out.append(o)
        s = ''.join(out)
        words.append(s if s else 'a')
    return ' '.join(words)

def gen_arabizi(qdict, level, seed=SEED):
    pd_, pr_ = LEVELS[level]
    return {q: to_arabizi(t, random.Random(f"{seed}-{level}-{q}"), pd_, pr_) for q, t in qdict.items()}

## 4) المحوّل العكسي بقيد معجمي
نولّد التهجئات الممكنة لكل كلمة Arabizi، ونُبقي **فقط** ما يوجد في مفردات المدوّنة، مرتبةً بالتكرار. جدول `DEROM` مكتوب يدويًا ومنفصل عن المولّد (وبدائله مختلفة قليلًا).
المرشح رقم j للاستعلام = المرشح j لكل كلمة (طريقة خشنة لكنها كافية للنسخة الأولية).

In [ ]:
ARABIC_WORD = re.compile(r'^[\u0621-\u064A]+$')
vocab_freq, prefix_freq = Counter(), Counter()
for toks in doc_tokens:
    for w in toks:
        if ARABIC_WORD.match(w): vocab_freq[w] += 1
for w, c in vocab_freq.items():
    for i in range(1, len(w) + 1):
        prefix_freq[w[:i]] += c

DEROM = {
 'aa':['ا'], 'ee':['ي'], 'oo':['و'], 'ou':['و'], 'sh':['ش'], 'ch':['ش'], 'kh':['خ'],
 'gh':['غ'], 'th':['ث','ذ'], 'dh':['ذ','ظ'],
 'a':['ا','','ع','ه'], 'e':['ا','','ي'], 'i':['ي','','ا'], 'o':['و','','ا'], 'u':['و',''],
 'b':['ب'], 't':['ت','ط'], 'j':['ج'], 'g':['ج','غ','ق'], 'h':['ه','ح'], 'd':['د','ض'],
 'r':['ر'], 'z':['ز','ظ','ذ'], 's':['س','ص','ث'], 'f':['ف'], 'q':['ق'], 'k':['ك'],
 'l':['ل'], 'm':['م'], 'n':['ن'], 'w':['و'], 'y':['ي'], 'v':['ف'], 'p':['ب'],
 '3':['ع'], '7':['ح'], '5':['خ'], '2':['ء','ا','ق'], '9':['ق','ص'], '6':['ط'], '8':['غ','ق'],
}
KEYS = set(DEROM)

def segment(w):
    out, i = [], 0
    while i < len(w):
        for L in (2, 1):
            piece = w[i:i + L]
            if piece in KEYS: out.append(piece); i += L; break
        else:
            out.append(w[i]); i += 1
    return out

def derom_word(w, top=5, beam=60, use_lexicon=True, pre=''):
    beams = [pre]
    for p in segment(w):
        opts = DEROM.get(p, [p])
        beams = list(dict.fromkeys(b + o for b in beams for o in opts))
        if use_lexicon:
            beams = sorted((b for b in beams if b in prefix_freq), key=lambda b: -prefix_freq[b])[:beam]
        else:
            beams = beams[:beam]
    if use_lexicon:
        words = sorted((b for b in beams if b in vocab_freq), key=lambda b: -vocab_freq[b])
    else:
        words = [b for b in beams if b]
    return words[:top]

ARTICLES = {"el", "al", "il", "l"}      # أداة التعريف المكتوبة منفصلة: el watan = الوطن

def derom_query(q, k=K_CANDIDATES, use_lexicon=True):
    toks = tok(q); per = []; i = 0
    while i < len(toks):
        w = toks[i]
        if w in ARTICLES and i + 1 < len(toks):          # نلصق ال بالكلمة التالية
            withal = derom_word(toks[i + 1], top=k, use_lexicon=use_lexicon, pre="ال")
            bare = derom_word(toks[i + 1], top=k, use_lexicon=use_lexicon)
            c = (withal + [x for x in bare if x not in withal])[:k]
            per.append(c if c else [toks[i + 1]])
            i += 2; continue
        c = derom_word(w, top=k, use_lexicon=use_lexicon)
        per.append(c if c else [w])                      # كلمة غير معروفة: نتركها كما هي
        i += 1
    if not per: return [q]
    cands = [" ".join(c[j] if j < len(c) else c[0] for c in per) for j in range(k)]
    return list(dict.fromkeys(cands))

def repaired_search(system, q, repair, k=K_CANDIDATES, use_lexicon=True):
    if repair == "raw": return search(system, q)
    cands = derom_query(q, k=(1 if repair == "R1" else k), use_lexicon=use_lexicon)
    if repair == "R1": return search(system, cands[0])
    runs = [search(system, c) for c in cands]
    if repair == "R4": runs.append(search(system, q))
    return rrf(runs)

## 5) مجموعتكِ البشرية (Q2 وQ3) — مهمتكِ اليدوية
شغّلي الخلية لإنشاء `human_set.csv`، ثم افتحيه في Google Sheets واملئي العمودين:
- `dialect_arabic`: الاستعلام بلهجتكِ بحروف عربية.
- `dialect_arabizi`: نفس الاستعلام بالأرقام واللاتينية كما تكتبين على هاتفكِ.

**قواعد:** اكتبي دون النظر إلى الأصل الفصيح قدر الإمكان. إن تغيّر المعنى حتى لم تعد الفقرة الصحيحة تجيب عنه، فاترك الصف فارغًا. ثم ارفعي الملف المعبّأ إلى نفس المجلد.

In [ ]:
HUMAN_CSV = os.path.join(OUT, "human_set.csv")
hq_ids = sorted(queries)[:N_HUMAN]
if not os.path.exists(HUMAN_CSV):
    pd.DataFrame({"qid": hq_ids, "arabic_original": [queries[q] for q in hq_ids],
                  "dialect_arabic": "", "dialect_arabizi": ""}).to_csv(HUMAN_CSV, index=False, encoding="utf-8-sig")
    print("أُنشئ القالب:", HUMAN_CSV, "- املئيه ثم أعيدي تشغيل الخلية")
hdf = pd.read_csv(HUMAN_CSV, dtype=str).fillna("")
hdf = hdf[(hdf.dialect_arabic.str.strip() != "") & (hdf.dialect_arabizi.str.strip() != "")]
human_ids = list(hdf.qid)
Q2 = dict(zip(hdf.qid, hdf.dialect_arabic)); Q3 = dict(zip(hdf.qid, hdf.dialect_arabizi))
print("استعلامات بشرية جاهزة:", len(human_ids))

استعلامات بشرية جاهزة: 60


## 6) التجارب
الحالات: Q0 (أصلي) | Q1_L1..L3 (Arabizi مولَّد) | Q2 وQ3 (بشري، على المجموعة الفرعية فقط).
الإصلاحات: raw (بلا) | R1 (مرشح واحد) | R3 (k مرشحات + RRF) | R4 (R3 + الخام).
لكل استعلام نحسب nDCG@10، ثم الفروق مع bootstrap (10,000) وتصحيح Holm.

In [ ]:
SYSTEMS = ["bm25", "dense", "hybrid"] if USE_DENSE else ["bm25"]
REPAIRS = ["raw", "R1", "R3", "R4"]

def per_query(qtexts, ids, system, repair, **kw):
    return np.array([ndcg10(repaired_search(system, qtexts[q], repair, **kw), qrels[q]) for q in ids])

all_ids = sorted(queries)
conds = {"Q0": (queries, all_ids)}
for L in (1, 2, 3): conds[f"Q1_L{L}"] = (gen_arabizi(queries, L), all_ids)
if human_ids:
    conds["Q2"] = (Q2, human_ids); conds["Q3"] = (Q3, human_ids)

scores = {}          # (cond, system, repair) -> array
for cname, (qt, ids) in conds.items():
    for s in SYSTEMS:
        for r in (["raw"] if cname in ("Q0", "Q2") else REPAIRS):   # الإصلاح لا معنى له للحروف العربية
            scores[(cname, s, r)] = per_query(qt, ids, s, r)
    print("done", cname)

q0 = {s_: dict(zip(all_ids, scores[("Q0", s_, "raw")])) for s_ in SYSTEMS}
rows = []
for (c, s_, r), v in scores.items():
    ids = conds[c][1]
    rows.append({"cond": c, "system": s_, "repair": r, "n": len(ids), "nDCG@10": v.mean(),
                 "Q0_same_queries": float(np.mean([q0[s_][q] for q in ids]))})
res = pd.DataFrame(rows)
res["gap_vs_Q0"] = res["nDCG@10"] - res["Q0_same_queries"]
raw_map = {(x.cond, x.system): x["nDCG@10"] for _, x in res[res.repair == "raw"].iterrows()}
def recovery(x):
    raw = raw_map.get((x.cond, x.system)); full = x.Q0_same_queries
    return np.nan if (x.repair == "raw" or full - raw < 1e-9) else (x["nDCG@10"] - raw) / (full - raw)
res["gap_recovered"] = res.apply(recovery, axis=1)
res = res.round(4); res.to_csv(os.path.join(OUT, "results_main.csv"), index=False)
res

done Q0
done Q1_L1
done Q1_L2
done Q1_L3
done Q2
done Q3


,cond,system,repair,n,nDCG@10,Q0_same_queries,gap_vs_Q0,gap_recovered
0,Q0,bm25,raw,200,0.6214,0.6214,0.0000,NaN
1,Q0,dense,raw,200,0.8524,0.8524,0.0000,NaN
2,Q0,hybrid,raw,200,0.7717,0.7717,0.0000,NaN
3,Q1_L1,bm25,raw,200,0.0016,0.6214,-0.6198,NaN
4,Q1_L1,bm25,R1,200,0.3575,0.6214,-0.2639,0.5742
5,Q1_L1,bm25,R3,200,0.3635,0.6214,-0.2579,0.5839
6,Q1_L1,bm25,R4,200,0.3604,0.6214,-0.2610,0.5789
7,Q1_L1,dense,raw,200,0.0000,0.8524,-0.8524,NaN
8,Q1_L1,dense,R1,200,0.2963,0.8524,-0.5562,0.3475
9,Q1_L1,dense,R3,200,0.2433,0.8524,-0.6091,0.2854


## 7) الدلالة الإحصائية (الإصلاح مقابل الخام)

In [ ]:
tests = []
for c in [k for k in conds if k not in ("Q0", "Q2")]:
    for s in SYSTEMS:
        raw = scores[(c, s, "raw")]
        for r in ("R1", "R3", "R4"):
            m, ci, p = paired_bootstrap(scores[(c, s, r)], raw)
            tests.append({"cond": c, "system": s, "repair": r, "mean_diff": m, "ci_lo": ci[0], "ci_hi": ci[1], "p": p})
tdf = pd.DataFrame(tests); tdf["p_holm"] = holm(tdf.p.values)
tdf.round(4).to_csv(os.path.join(OUT, "results_significance.csv"), index=False); tdf.round(4)

,cond,system,repair,mean_diff,ci_lo,ci_hi,p,p_holm
0,Q1_L1,bm25,R1,0.3559,0.3043,0.4070,0.0000,0.0000
1,Q1_L1,bm25,R3,0.3620,0.3097,0.4138,0.0000,0.0000
2,Q1_L1,bm25,R4,0.3588,0.3069,0.4097,0.0000,0.0000
3,Q1_L1,dense,R1,0.2963,0.2429,0.3508,0.0000,0.0000
4,Q1_L1,dense,R3,0.2433,0.1944,0.2938,0.0000,0.0000
5,Q1_L1,dense,R4,0.1983,0.1535,0.2449,0.0000,0.0000
6,Q1_L1,hybrid,R1,0.4116,0.3567,0.4654,0.0000,0.0000
7,Q1_L1,hybrid,R3,0.3855,0.3334,0.4386,0.0000,0.0000
8,Q1_L1,hybrid,R4,0.3130,0.2662,0.3615,0.0000,0.0000
9,Q1_L2,bm25,R1,0.1867,0.1435,0.2317,0.0000,0.0000


## 8) Ablation (على Q1_L2، وعلى Q3 إن وُجدت)

In [ ]:
abl = []
target = [c for c in ("Q1_L2", "Q3") if c in conds]
for c in target:
    qt, ids = conds[c]
    for s in SYSTEMS:
        for k in (1, 3, 5, 10):
            abl.append({"cond": c, "system": s, "variant": f"R3 k={k}", "nDCG@10": per_query(qt, ids, s, "R3", k=k).mean()})
        abl.append({"cond": c, "system": s, "variant": "R3 k=5 بلا قيد معجمي",
                    "nDCG@10": per_query(qt, ids, s, "R3", k=5, use_lexicon=False).mean()})
adf = pd.DataFrame(abl).round(4); adf.to_csv(os.path.join(OUT, "results_ablation.csv"), index=False); adf

,cond,system,variant,nDCG@10
0,Q1_L2,bm25,R3 k=1,0.1882
1,Q1_L2,bm25,R3 k=3,0.1915
2,Q1_L2,bm25,R3 k=5,0.1855
3,Q1_L2,bm25,R3 k=10,0.1853
4,Q1_L2,bm25,R3 k=5 بلا قيد معجمي,0.0797
5,Q1_L2,dense,R3 k=1,0.1694
6,Q1_L2,dense,R3 k=3,0.1113
7,Q1_L2,dense,R3 k=5,0.1130
8,Q1_L2,dense,R3 k=10,0.1130
9,Q1_L2,dense,R3 k=5 بلا قيد معجمي,0.0794


## 9) تحليل لكل استعلام + عيّنة لتصنيف الأخطاء
نصنّف يدويًا أسوأ الحالات في عمود `error_type`: التباس حرف | حذف حركات | كلمة لهجية غير موجودة | اسم أجنبي | خلط لغات | أخرى.

In [ ]:
c = "Q3" if "Q3" in conds else "Q1_L2"
qt, ids = conds[c]
sysname = "hybrid" if USE_DENSE else "bm25"
a, b = scores[(c, sysname, "raw")], scores[(c, sysname, "R3")]
o = np.array([q0[sysname][q] for q in ids])      # Q0 على نفس الاستعلامات
print(f"[{c}/{sysname}] أفضل بعد الإصلاح: {(b>a).sum()} | أسوأ: {(b<a).sum()} | بلا تغيّر: {(b==a).sum()}")
fail = [(i, q) for i, q in enumerate(ids) if b[i] < o[i]]
fail.sort(key=lambda t: o[t[0]] - b[t[0]], reverse=True)
err = pd.DataFrame([{"qid": q, "original": queries[q], "written": qt[q], "top_candidate": derom_query(qt[q])[0],
                     "nDCG_Q0": round(o[i], 3), "nDCG_raw": round(a[i], 3), "nDCG_R3": round(b[i], 3), "error_type": ""}
                    for i, q in fail[:40]])
err.to_csv(os.path.join(OUT, "error_analysis_to_label.csv"), index=False, encoding="utf-8-sig"); err.head(15)

[Q3/hybrid] أفضل بعد الإصلاح: 15 | أسوأ: 4 | بلا تغيّر: 41


,qid,original,written,top_candidate,nDCG_Q0,nDCG_raw,nDCG_R3,error_type
0,10761,ما هي منظمة فريدم هاوس؟,shu hiye monazzamet freedom house?,شو هي monazzamet فريدم هوس,1.000,0.0,0.0,
1,11095,هل ستحل السيارات الكهربائية محل السيارات العاد...,ya tara el sayyarat el kahraba2iyye ra7 te7el ...,يا تاره sayyarat الكهربايي راح تحل مكان sayyar...,1.000,0.0,0.0,
2,11164,ما هي الكا ؟,shu hiye el Ka?,شو هي الكا,1.000,0.0,0.0,
3,12104,متي توفي يوربيديس ؟,emta mat yuribidis?,يمت مات يوربيديس,1.000,0.0,0.0,
4,12714,ما هي الكيمياء الكمومية؟,shu hiye el kimya2 el kammiyye?,شو هي الكيمياء kammiyye,1.000,0.0,0.0,
5,13617,أين تقع مدينة كن تاه؟,wein bet2a3 madinet kan tah?,وان bet2a3 madinet كان طه,1.000,0.0,0.0,
6,13691,"متى أطلقت فرقة صودا ستيريوا أغنية ""نادا بيرسون...",emta tal3at ferqet soda stereo oghniyet nada p...,يمت طلعت فرقت سد ستار اغنيات نهضه بيرسونال,1.000,0.0,0.0,
7,13864,ما هو الشره الرقمي؟,shu huwwe el sharah el raqami?,شو هووي الشهره الرقم,1.000,0.0,0.0,
8,10717,كم عدد الأبراج الفلكية ؟,2adesh 3adad el abraj el falakiyye?,قادش عدد البرج falakiyye,0.984,0.0,0.0,
9,12224,ما هي عاصمة كوريا الجنوبية؟,shu 3asmet koriya el janubiyye?,شو عصمت كوريا janubiyye,0.967,0.0,0.0,


## 10) (اختياري) مرشحات من نموذج لغوي
مرّري دالة `llm_fn(prompt)->str` تستدعي أي نموذج (مفتوح أو API) ثم أضيفي ناتجها كمرشحات في `repaired_search`. استخدمي **نموذجًا مختلفًا** عن الذي ولّد البيانات، وسجّلي اسمه وإصداره وتاريخ الاستخدام.

In [ ]:
import json as _json
PROMPT = ("حوّل الاستعلام التالي المكتوب بالحروف اللاتينية (Arabizi) إلى العربية. "
          "أعد {k} احتمالات مرتبة من الأرجح إلى الأقل كمصفوفة JSON من النصوص فقط، بلا شرح.\n"
          "الاستعلام: {q}")
def llm_candidates(q, llm_fn, k=3):
    try:
        txt = llm_fn(PROMPT.format(k=k, q=q))
        arr = _json.loads(txt[txt.index("["): txt.rindex("]") + 1])
        return [str(x) for x in arr][:k]
    except Exception:
        return []          # عند الفشل نعود للخام